<a href="https://colab.research.google.com/github/BekenAidana/AI_agents/blob/main/agentloopwithfunctioncalling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!!pip install litellm

# Important!!!
#
# <---- Set your 'OPENAI_API_KEY' as a secret over there with the "key" icon
#
#
import os
from google.colab import userdata
api_key = userdata.get('GEMINI_API_KEY')
os.environ['GEMINI_API_KEY'] = api_key

In [ ]:
import json
import os
from typing import List

from litellm import completion

def list_files(path: str = ".") -> List[str]:
    """
    List files in the specified directory.
    Defaults to the current directory.
    """
    return os.listdir(path)

def read_file(file_name: str) -> str:
    """Read a file's contents."""
    try:
        with open(file_name, "r") as file:
            return file.read()
    except FileNotFoundError:
        return f"Error: {file_name} not found."
    except Exception as e:
        return f"Error: {str(e)}"

def write_to_file(path: str, content: str) -> str:
    """
    Writes content to a file strictly within the 'sample_data/output' directory.
    """
    BASE_DIR = os.path.abspath("sample_data")
    target_path = os.path.abspath(os.path.join(BASE_DIR, path))

    if not target_path.startswith(BASE_DIR):
        return f"Error: Access denied. You can only write to {BASE_DIR}"

    try:
        os.makedirs(os.path.dirname(target_path), exist_ok=True)
        with open(target_path, 'w', encoding='utf-8') as f:
            f.write(content)
        return f"Successfully written to {target_path}"
    except Exception as e:
        return f"Error writing to file: {str(e)}"

def terminate(message: str) -> None:
    """Terminate the agent loop and provide a summary message."""
    print(f"Termination message: {message}")

tool_functions = {
    "list_files": list_files,
    "read_file": read_file,
    "terminate": terminate
}

tools = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "Returns a list of files in the specified directory. Defaults to the current directory if no path is provided.",
            "parameters": {"type": "object",
                           "properties": {"path": {"type":"string"}},
                           "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Reads the content of a specified file in the directory.",
            "parameters": {
                "type": "object",
                "properties": {
                    "file_name": {"type": "string",
                                  "description": "The path to the directory to list files from. Use '.' for the current directory."}
                    },
                "required": ["file_name"]
            }
        }
    },
    {
    "type": "function",
    "function": {
        "name": "write_to_file",
        "description": "Saves content to a file. NOTE: For security, you can only write files inside the 'sample_data' directory.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {
                    "type": "string",
                    "description": "The name or sub-path of the file (e.g., 'results.txt' or 'logs/analysis.md'). It will be saved inside 'sample_data/'."
                },
                "content": {
                    "type": "string",
                    "description": "The text content to write."
                }
            },
            "required": ["path", "content"]
        }
    }
},
    {
        "type": "function",
        "function": {
            "name": "terminate",
            "description": "Terminates the conversation. No further actions or interactions are possible after this. Prints the provided message for the user.",
            "parameters": {
                "type": "object",
                "properties": {
                    "message": {"type": "string"},
                },
                "required": ["message"]
            }
        }
    }
]

agent_rules = [{
    "role": "system",
    "content": """
You are an AI agent that can perform tasks by using available tools.

If a user asks about files, documents, or content, first list the files before reading them.

When you are done, terminate the conversation by using the "terminate" tool and I will provide the results to the user.
"""
}]

In [ ]:
def new_chat():
  iterations = 0
  max_iterations = 10

  user_task = input("What would you like me to do? ")

  memory = [{"role": "user", "content": user_task}]

  while iterations < max_iterations:

      messages = agent_rules + memory

      response = completion(
          model="gemini/gemini-2.5-flash",
          messages=messages,
          tools=tools,
          max_tokens=1024
      )

      if response.choices[0].message.tool_calls:
          tool = response.choices[0].message.tool_calls[0]
          tool_name = tool.function.name
          tool_args = json.loads(tool.function.arguments)

          action = {
              "tool_name": tool_name,
              "args": tool_args
          }

          if tool_name == "terminate":
              print(f"Termination message: {tool_args['message']}")
              break
          elif tool_name in tool_functions:
              try:
                  result = {"result": tool_functions[tool_name](**tool_args)}
              except Exception as e:
                  result = {"error":f"Error executing {tool_name}: {str(e)}"}
          else:
              result = {"error": f"Unknown tool: {tool_name}"}

          print(f"Executing: {tool_name} with args {tool_args}")
          print(f"Result: {result}")
          memory.extend([
              {"role": "assistant", "content": json.dumps(action)},
              {"role": "user", "content": json.dumps(result)}
          ])
      else:
          result = response.choices[0].message.content
          print(f"Response: {result}")
          break

In [ ]:
new_chat()

What would you like me to do? Что ты умеешь делать?
Response: Я умею работать с файлами:
- Просматривать список файлов в указанной директории.
- Читать содержимое файлов.
- Записывать содержимое в файлы (только в директорию 'sample_data').
- А также завершать диалог.


In [ ]:
new_chat()

What would you like me to do? Что находится в тек директории

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



RateLimitError: litellm.RateLimitError: litellm.RateLimitError: geminiException - {
  "error": {
    "code": 429,
    "message": "You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 14.156528516s.",
    "status": "RESOURCE_EXHAUSTED",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.Help",
        "links": [
          {
            "description": "Learn more about Gemini API quotas",
            "url": "https://ai.google.dev/gemini-api/docs/rate-limits"
          }
        ]
      },
      {
        "@type": "type.googleapis.com/google.rpc.QuotaFailure",
        "violations": [
          {
            "quotaMetric": "generati

In [ ]:
new_chat()

What would you like me to do? Какие есть файлы в директории?
Executing: list_files with args {}
Result: {'result': ['.config', 'sample_data']}
Response: Извините, в директории нет файлов, только папки: ".config" и "sample_data".


In [ ]:
new_chat()

What would you like me to do? Можешь ли ты мне прочитать папку sample_data
Response: Я не могу читать папки напрямую. Я могу только перечислить файлы в текущей директории. Хотите, я перечислю файлы для вас?


In [ ]:
new_chat()

What would you like me to do? Скажи какие есть папки в файле и допиши в конце название рандомной пиццы и цвета, а также в конце 47
Executing: list_files with args {}
Result: {'result': ['.config', 'sample_data']}
Response: В файле есть папки ".config" и "sample_data". Рандомная пицца: "Маргарита", цвет: "синий". 47.


In [ ]:
new_chat()

What would you like me to do? Скажи какие есть папки в файле и допиши к каждому ответу название рандомной пиццы и цвета, а также в конце 47
Executing: list_files with args {}
Result: {'result': ['.config', 'sample_data']}
Termination message: .config: Маргарита Синий 47
sample_data: Пепперони Красный 47
